# Environment Check
**Run this notebook before the workshop starts. Do not skip it.**

It validates five things:
1. Python version
2. Required package imports and versions
3. Presence and schema of `data/m5_workshop_subset.parquet`
4. Required directory structure (`artifacts/`, `params/`, `src/`)
5. Artifact registry readiness (precomputed Red Path files)

Every check either prints `✓ PASS` or raises a hard error with a fix instruction.
**All five checks must pass before the session begins.**

This notebook makes no network calls. It does not touch S3.

In [1]:
# =============================================================================
# CELL 1 — Path setup
# Ensures the project root is on sys.path so config.py and src/ resolve
# correctly whether running locally or on Colab.
# =============================================================================
import sys
from pathlib import Path

# Resolve project root: the directory that contains this notebook
PROJECT_ROOT = Path().resolve()

# If running on Colab and the repo was cloned to a non-default path,
# update PROJECT_ROOT manually:
#   PROJECT_ROOT = Path("/content/packt-modern-time-series")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\tacke\Documents\packt-modern-time-series


In [2]:
# =============================================================================
# CELL 2 — CHECK 1: Python version
# Workshop is tested on Python 3.12. Warn on 3.13+. Block on <3.10.
# =============================================================================
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # Prevent OpenMP crash on Windows

REQUIRED_MAJOR = 3
REQUIRED_MINOR = 10
TESTED_MINOR   = 12

major = sys.version_info.major
minor = sys.version_info.minor
version_str = f"{major}.{minor}.{sys.version_info.micro}"

if major < REQUIRED_MAJOR or (major == REQUIRED_MAJOR and minor < REQUIRED_MINOR):
    raise EnvironmentError(
        f"\n"
        f"  ✗ FAIL — Python version: {version_str}\n"
        f"  Required: Python >= 3.10\n"
        f"  Fix: upgrade your Python environment or switch to the correct Colab runtime."
    )
elif minor > TESTED_MINOR:
    print(
        f"  ⚠ WARN — Python {version_str} detected. Workshop tested on 3.12.\n"
        f"  Most functionality will work, but report any unexpected errors."
    )
else:
    print(f"  ✓ PASS — Python {version_str}")

  ✓ PASS — Python 3.12.12


In [3]:
# =============================================================================
# CELL 3 — CHECK 2: Package imports
# Attempts to import every required package.
# Fails on missing packages. No version pinning — any installed version is fine.
# =============================================================================

REQUIRED_PACKAGES = [
    # (import_name, pip_name)
    ("pandas",         "pandas"),
    ("numpy",          "numpy"),
    ("pyarrow",        "pyarrow"),
    ("matplotlib",     "matplotlib"),
    ("statsforecast",  "statsforecast"),
    ("mlforecast",     "mlforecast"),
    ("neuralforecast", "neuralforecast"),
    ("lightgbm",       "lightgbm"),
    ("sklearn",        "scikit-learn"),
    ("torch",          "torch"),
    ("tqdm",           "tqdm"),
]

import importlib
import importlib.metadata

all_passed = True
results = []

for import_name, pip_name in REQUIRED_PACKAGES:
    try:
        mod = importlib.import_module(import_name)
        try:
            installed = importlib.metadata.version(pip_name)
        except importlib.metadata.PackageNotFoundError:
            installed = getattr(mod, "__version__", "unknown")
        results.append((True, f"  ✓ PASS — {pip_name}=={installed}"))
    except ImportError:
        results.append((False,
            f"  ✗ FAIL — {pip_name} not found.\n"
            f"    Fix: see README setup instructions for your platform."
        ))
        all_passed = False

for _, msg in results:
    print(msg)

if not all_passed:
    raise EnvironmentError(
        "\n  ✗ One or more required packages are missing.\n"
        "  See README.md Setup section for your platform (Colab / Windows / macOS)."
    )

print("\n  ✓ PASS — All packages imported successfully.")

c:\Users\tacke\anaconda3\envs\packt_timeseries_cohort\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


  ✓ PASS — pandas==2.3.3
  ✓ PASS — numpy==2.4.2
  ✓ PASS — pyarrow==23.0.1
  ✓ PASS — matplotlib==3.10.8
  ✓ PASS — statsforecast==2.0.3
  ✓ PASS — mlforecast==1.0.31
  ✓ PASS — neuralforecast==3.1.5
  ✓ PASS — lightgbm==4.6.0
  ✓ PASS — scikit-learn==1.8.0
  ✓ PASS — torch==2.10.0
  ✓ PASS — tqdm==4.67.3

  ✓ PASS — All packages imported successfully.


In [4]:
# =============================================================================
# CELL 4 — CHECK 3: Directory structure
# Validates that required repo directories exist.
# A missing directory means the repo was cloned or unzipped incorrectly.
# =============================================================================

REQUIRED_DIRS = [
    "src",
    "artifacts",
    "params",
    "data",
    "companion_asset",
    "instructor_notebooks",
    "student_notebooks",
]

dir_failures = []

for d in REQUIRED_DIRS:
    full_path = PROJECT_ROOT / d
    if full_path.is_dir():
        print(f"  ✓ PASS — {d}/")
    else:
        print(f"  ✗ FAIL — {d}/ not found at {full_path}")
        dir_failures.append(d)

if dir_failures:
    raise EnvironmentError(
        f"\n  ✗ Missing directories: {dir_failures}\n"
        f"  Fix: ensure you are running from the project root ('{PROJECT_ROOT}')\n"
        f"  and that the repository was fully cloned/unzipped."
    )

print("\n  ✓ PASS — All required directories present.")

  ✓ PASS — src/
  ✓ PASS — artifacts/
  ✓ PASS — params/
  ✓ PASS — data/
  ✓ PASS — companion_asset/
  ✓ PASS — instructor_notebooks/
  ✓ PASS — student_notebooks/

  ✓ PASS — All required directories present.


In [5]:
# =============================================================================
# CELL 5 — CHECK 4: Workshop data subset
# Validates presence, schema, row count, and date range of
# data/m5_workshop_subset.parquet.
#
# This file ships with the repository. If it is missing, the repo was not
# cloned correctly or the file was accidentally deleted.
# =============================================================================
import pandas as pd
from config import (
    WORKSHOP_SUBSET_PATH,
    WORKSHOP_SUBSET_N,
    MIN_HISTORY_DAYS,
    HORIZON,
    N_WINDOWS,
    STEP_SIZE,
)

# --- 4a. File existence ---
if not WORKSHOP_SUBSET_PATH.exists():
    raise FileNotFoundError(
        f"\n  ✗ FAIL — Workshop data not found: {WORKSHOP_SUBSET_PATH}\n"
        f"  This file ships with the repository.\n"
        f"  Fix: re-clone the repo or run: git pull\n"
        f"  If the file is still missing, contact the instructor."
    )

print(f"  ✓ PASS — File exists: {WORKSHOP_SUBSET_PATH.name}")

# --- 4b. Load and schema check ---
df = pd.read_parquet(WORKSHOP_SUBSET_PATH)

REQUIRED_COLUMNS = {"unique_id": "object", "ds": "datetime64[ns]", "y": "float64"}
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]

if missing_cols:
    raise ValueError(
        f"\n  ✗ FAIL — Missing columns in subset: {missing_cols}\n"
        f"  Found columns: {list(df.columns)}\n"
        f"  Fix: re-clone the repo to get the correct data file."
    )

df["ds"] = pd.to_datetime(df["ds"])
df["y"]  = df["y"].astype("float64")
print("  ✓ PASS — Schema columns present: unique_id, ds, y")

# --- 4c. Series count ---
n_series = df["unique_id"].nunique()
MIN_ACCEPTABLE_SERIES = int(WORKSHOP_SUBSET_N * 0.95)

if n_series < MIN_ACCEPTABLE_SERIES:
    raise ValueError(
        f"\n  ✗ FAIL — Only {n_series} series found. Expected ~{WORKSHOP_SUBSET_N}.\n"
        f"  Fix: re-clone the repo to get the correct data file."
    )

print(f"  ✓ PASS — Series count: {n_series:,} (target: {WORKSHOP_SUBSET_N:,})")

# --- 4d. Row count ---
n_rows = len(df)
print(f"  ✓ PASS — Total rows: {n_rows:,}")

# --- 4e. Date range ---
min_date = df["ds"].min()
max_date = df["ds"].max()
date_range_days = (max_date - min_date).days
min_required_days = MIN_HISTORY_DAYS + (N_WINDOWS * STEP_SIZE) + HORIZON

if date_range_days < min_required_days:
    raise ValueError(
        f"\n  ✗ FAIL — Date range too short: {date_range_days} days.\n"
        f"  Required: >= {min_required_days} days\n"
        f"  Fix: re-clone the repo to get the correct data file."
    )

print(f"  ✓ PASS — Date range: {min_date.date()} → {max_date.date()} ({date_range_days} days)")

# --- 4f. Null check ---
null_counts = df[["unique_id", "ds", "y"]].isnull().sum()
if null_counts.any():
    raise ValueError(
        f"\n  ✗ FAIL — Null values detected in subset:\n{null_counts[null_counts > 0]}\n"
        f"  Fix: re-clone the repo to get the correct data file."
    )

print("  ✓ PASS — No null values in required columns.")
print(f"\n  ✓ PASS — Data subset is valid.")

  ✓ PASS — File exists: m5_workshop_subset.parquet
  ✓ PASS — Schema columns present: unique_id, ds, y
  ✓ PASS — Series count: 1,000 (target: 1,000)
  ✓ PASS — Total rows: 1,941,000
  ✓ PASS — Date range: 2011-01-29 → 2016-05-22 (1940 days)
  ✓ PASS — No null values in required columns.

  ✓ PASS — Data subset is valid.


In [6]:
# =============================================================================
# CELL 6 — CHECK 5: Artifact registry readiness (Red Path)
# Checks whether precomputed artifacts exist for the Red Path recovery.
# Missing artifacts are a WARNING, not a hard error — the workshop can
# technically proceed on the Green Path only.
# But if you are the instructor, all artifacts must be present before the session.
# =============================================================================
from src.checkpointing import list_checkpoints
from config import ARTIFACT_REGISTRY

# Parquet/CSV artifacts only — skip pkl and json which are generated live
PRECOMPUTED_ARTIFACTS = [
    "03_validated_panel",
    "04_baseline_forecasts",
    "04_baseline_cv_scores",
    "05_ml_forecasts",
    "06_dl_forecasts",
    "07_uncertainty_leaderboard",
    "08_final_master_leaderboard",
]

missing_artifacts = []
for name in PRECOMPUTED_ARTIFACTS:
    path = ARTIFACT_REGISTRY[name]
    if path.exists():
        print(f"  ✓ EXISTS  — {name}")
    else:
        print(f"  ✗ MISSING — {name}")
        missing_artifacts.append(name)

if missing_artifacts:
    print(
        f"\n  ⚠ WARN — {len(missing_artifacts)} precomputed artifact(s) missing.\n"
        f"  The workshop Green Path will still work, but Red Path recovery cells\n"
        f"  will fail for: {missing_artifacts}\n"
        f"  Fix: run src/build_offline_artifacts.py before the session."
    )
else:
    print("\n  ✓ PASS — All precomputed artifacts present. Red Path is ready.")

  ✗ MISSING — 03_validated_panel
  ✗ MISSING — 04_baseline_forecasts
  ✗ MISSING — 04_baseline_cv_scores
  ✗ MISSING — 05_ml_forecasts
  ✗ MISSING — 06_dl_forecasts
  ✗ MISSING — 07_uncertainty_leaderboard
  ✗ MISSING — 08_final_master_leaderboard

  ⚠ WARN — 7 precomputed artifact(s) missing.
  The workshop Green Path will still work, but Red Path recovery cells
  will fail for: ['03_validated_panel', '04_baseline_forecasts', '04_baseline_cv_scores', '05_ml_forecasts', '06_dl_forecasts', '07_uncertainty_leaderboard', '08_final_master_leaderboard']
  Fix: run src/build_offline_artifacts.py before the session.


In [7]:
# =============================================================================
# CELL 7 — Final summary and save status artifact
# Writes artifacts/00_env_status.json so other notebooks can confirm
# the env check was run. Also gives the instructor a clear go/no-go.
# =============================================================================
import json
from datetime import datetime, timezone
from config import ARTIFACT_DIR

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

status = {
    "env_check_passed": True,
    "timestamp_utc":    datetime.now(timezone.utc).isoformat(),
    "python_version":   version_str,
    "n_series":         int(n_series),
    "n_rows":           int(n_rows),
    "date_range": {
        "min": str(min_date.date()),
        "max": str(max_date.date()),
    },
    "missing_artifacts": missing_artifacts,
}

status_path = ARTIFACT_DIR / "00_env_status.json"
with open(status_path, "w") as f:
    json.dump(status, f, indent=2)

print("=" * 55)
print("  ENVIRONMENT CHECK COMPLETE")
print("=" * 55)
print(f"  Python      : {version_str}")
print(f"  Series      : {n_series:,}")
print(f"  Rows        : {n_rows:,}")
print(f"  Date range  : {min_date.date()} → {max_date.date()}")
if missing_artifacts:
    print(f"  Red Path    : ⚠ {len(missing_artifacts)} artifact(s) missing")
else:
    print(f"  Red Path    : ✓ Ready")
print(f"  Status file : {status_path.name}")
print("=" * 55)
print("  ✓ GO — Environment is ready for the workshop.")
print("=" * 55)

  ENVIRONMENT CHECK COMPLETE
  Python      : 3.12.12
  Series      : 1,000
  Rows        : 1,941,000
  Date range  : 2011-01-29 → 2016-05-22
  Red Path    : ⚠ 7 artifact(s) missing
  Status file : 00_env_status.json
  ✓ GO — Environment is ready for the workshop.
